In [1]:
import os
import json
import datetime
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()
import warnings
warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["TRANSFORMERS_VERBOSITY"] = "error"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"


data_dir = r"C:\Users\X\Desktop\X\ALTA\alta2026_public_data"
root_dir = r"C:\Users\X\Desktop\X\ALTA"

model_name = "roberta-base"
epochs = 6
batch_size = 16
lr = 2e-5
max_length = 256
patience = 2


run_id = datetime.datetime.now().strftime("%Y%m%d-%H%M%S") + f"_{model_name}"
run_dir = os.path.join(root_dir, "runs", run_id)
work_dir = os.path.join(run_dir, "tf_runs")
model_out_dir = os.path.join(run_dir, "final_models")
metrics_dir = os.path.join(run_dir, "metrics")
out = os.path.join(run_dir, "answer.csv")

for d in (run_dir, work_dir, model_out_dir, metrics_dir):
    os.makedirs(d, exist_ok=True)


run_config = {
    "run_id": run_id,
    "timestamp": datetime.datetime.now().isoformat(),
    "model_name": model_name,
    "epochs": epochs,
    "batch_size": batch_size,
    "lr": lr,
    "max_length": max_length,
    "patience": patience,
    "data_dir": data_dir,
}
with open(os.path.join(run_dir, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2)


class TextDataset(Dataset):
    def __init__(self, encodings, labels=None, sample_weights=None):
        self.encodings = encodings
        self.labels = labels
        self.sample_weights = sample_weights

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx])
        if self.sample_weights is not None:
            item["sample_weight"] = torch.tensor(self.sample_weights[idx], dtype=torch.float)
        return item


class WeightedCollator(DataCollatorWithPadding):
    def __call__(self, features):
        weights = [f.pop("sample_weight") for f in features if "sample_weight" in f]
        batch = super().__call__(features)
        if weights:
            batch["sample_weight"] = torch.stack(weights)
        return batch


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        weight = inputs.pop("sample_weight", None)
        outputs = model(**inputs)
        logits = outputs.logits
        per_sample = torch.nn.functional.cross_entropy(
            logits, labels, reduction="none"
        )
        if weight is not None:
            weight = weight.to(logits.device)
            loss = (per_sample * weight).sum() / weight.sum()
        else:
            loss = per_sample.mean()
        return (loss, outputs) if return_outputs else loss


def get_device():
    if torch.backends.mps.is_available():
        return "mps"
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def dialect_class_weights(labels, variety):
    labels = np.array(labels)
    variety = np.array(variety)
    weights = np.ones(len(labels), dtype=np.float32)

    for v in np.unique(variety):
        mask = variety == v
        cw = compute_class_weight(
            class_weight="balanced",
            classes=np.array([0, 1]),
            y=labels[mask]
        )
        for c in (0, 1):
            weights[mask & (labels == c)] = cw[c]

    return weights


def make_compute_metrics(variety_array):
    varieties = sorted(set(variety_array))

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        result = {
            "macro_f1": f1_score(labels, preds, average="macro")
        }

        per_variety = []

        for v in varieties:
            mask = variety_array == v
            f1 = f1_score(
                labels[mask],
                preds[mask],
                average="macro"
            )
            result[f"macro_f1_{v}"] = f1
            per_variety.append(f1)

        result["weak_macro_f1"] = min(per_variety)

        return result

    return compute_metrics


def train_one_task(task, train_df, valid_df, tokenizer, device):
    train_enc = tokenizer(
        list(train_df["text"]),
        truncation=True,
        max_length=max_length
    )

    valid_enc = tokenizer(
        list(valid_df["text"]),
        truncation=True,
        max_length=max_length
    )

    train_labels = train_df[task].tolist()

    sample_weights = dialect_class_weights(
        train_labels,
        train_df["variety"].tolist()
    )

    train_ds = TextDataset(
        train_enc,
        train_labels,
        sample_weights
    )

    valid_ds = TextDataset(
        valid_enc,
        valid_df[task].tolist()
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2
    )

    model.to(device)

    collator = WeightedCollator(tokenizer)

    training_args = TrainingArguments(
        output_dir=os.path.join(work_dir, task),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=lr,
        warmup_ratio=0.1,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="weak_macro_f1",
        greater_is_better=True,
        logging_steps=50,
        report_to=[]
    )

    trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
        compute_metrics=make_compute_metrics(
            valid_df["variety"].to_numpy()
        ),
        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=patience
            )
        ],
    )

    trainer.train()


    save_path = os.path.join(model_out_dir, task)
    os.makedirs(save_path, exist_ok=True)
    trainer.save_model(save_path)
    tokenizer.save_pretrained(save_path)


    with open(os.path.join(metrics_dir, f"{task}_trainer_log_history.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)

    preds = trainer.predict(valid_ds)

    return np.argmax(
        preds.predictions,
        axis=-1
    )


def plot_confusion_matrix(cm, class_names, title, save_path):
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_title(title)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks(range(len(class_names)))
    ax.set_yticks(range(len(class_names)))
    ax.set_xticklabels(class_names)
    ax.set_yticklabels(class_names)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                     color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150)
    plt.close(fig)


def compute_all_metrics(gold_df, run_df):
    """
    Computes the official ALTA 2026 scoring-program metric (weak macro F1,
    i.e. min-over-dialect macro F1 per task, averaged across tasks — as
    implemented in the shared-task scoring script), plus standard extra
    metrics (accuracy, precision, recall, per-class F1, confusion matrices)
    for every task/dialect combination.

    Note: I could not access the login-gated Codabench evaluation page
    directly, so the "official" metric here is taken from the scoring
    logic already present in the original training script (which mirrors
    the competition's scoring program). The additional metrics below are
    standard classification metrics, not necessarily part of the official
    leaderboard score.
    """
    all_metrics = {}
    official_scores = {}
    class_names = ["0", "1"]

    for task in ["sentiment", "sarcasm"]:
        all_metrics[task] = {}
        for dialect in ["en-AU", "en-UK"]:
            gold = gold_df[gold_df["variety"] == dialect][task]
            pred = run_df[run_df["variety"] == dialect][task]

            macro_f1 = f1_score(gold, pred, average="macro")
            macro_precision = precision_score(gold, pred, average="macro", zero_division=0)
            macro_recall = recall_score(gold, pred, average="macro", zero_division=0)
            acc = accuracy_score(gold, pred)
            report = classification_report(gold, pred, output_dict=True, zero_division=0)
            cm = confusion_matrix(gold, pred, labels=[0, 1])

            all_metrics[task][dialect] = {
                "accuracy": acc,
                "macro_f1": macro_f1,
                "macro_precision": macro_precision,
                "macro_recall": macro_recall,
                "classification_report": report,
                "confusion_matrix": cm.tolist(),
            }

            official_scores[f"{task}-{dialect}"] = macro_f1


            cm_df = pd.DataFrame(cm, index=[f"true_{c}" for c in class_names],
                                  columns=[f"pred_{c}" for c in class_names])
            cm_df.to_csv(os.path.join(metrics_dir, f"cm_{task}_{dialect}.csv"))
            plot_confusion_matrix(
                cm, class_names,
                title=f"{task} — {dialect}",
                save_path=os.path.join(metrics_dir, f"cm_{task}_{dialect}.png"),
            )


        gold_all = gold_df[task]
        pred_all = run_df[task]
        cm_all = confusion_matrix(gold_all, pred_all, labels=[0, 1])
        all_metrics[task]["overall"] = {
            "accuracy": accuracy_score(gold_all, pred_all),
            "macro_f1": f1_score(gold_all, pred_all, average="macro"),
            "macro_precision": precision_score(gold_all, pred_all, average="macro", zero_division=0),
            "macro_recall": recall_score(gold_all, pred_all, average="macro", zero_division=0),
            "classification_report": classification_report(gold_all, pred_all, output_dict=True, zero_division=0),
            "confusion_matrix": cm_all.tolist(),
        }
        cm_df_all = pd.DataFrame(cm_all, index=[f"true_{c}" for c in class_names],
                                  columns=[f"pred_{c}" for c in class_names])
        cm_df_all.to_csv(os.path.join(metrics_dir, f"cm_{task}_overall.csv"))
        plot_confusion_matrix(
            cm_all, class_names,
            title=f"{task} — overall",
            save_path=os.path.join(metrics_dir, f"cm_{task}_overall.png"),
        )


    final = (
        min(official_scores["sentiment-en-AU"], official_scores["sentiment-en-UK"])
        + min(official_scores["sarcasm-en-AU"], official_scores["sarcasm-en-UK"])
    ) / 2

    all_metrics["official_weak_macro_f1_per_task_dialect"] = official_scores
    all_metrics["official_final_score"] = final

    return all_metrics, final


device = get_device()
print(f"Run ID: {run_id}")
print(f"Outputs will be written under: {run_dir}")
print(f"Using device: {device}")

train_df = pd.read_csv(os.path.join(data_dir, "train.csv"))
valid_df = pd.read_csv(os.path.join(data_dir, "valid.csv"))

tokenizer = AutoTokenizer.from_pretrained(model_name)

preds = {}

for task in ["sentiment", "sarcasm"]:
    print(f"\nFine-tuning for task: {task}")
    preds[task] = train_one_task(task, train_df, valid_df, tokenizer, device)

answer_df = valid_df[["source", "variety", "text"]].copy()
answer_df["sentiment"] = preds["sentiment"]
answer_df["sarcasm"] = preds["sarcasm"]
answer_df.to_csv(out, index=False)
print(f"\nWrote predictions to {out}")

all_metrics, final = compute_all_metrics(valid_df, answer_df)

with open(os.path.join(metrics_dir, "all_metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2)

print("\nOfficial style scores (weak macro F1) on the public valid set:")
for k, v in all_metrics["official_weak_macro_f1_per_task_dialect"].items():
    print(f"  {k}: {v:.4f}")
print(f"  final score: {final:.4f}")

print(f"\nAll metrics, confusion matrices, and models saved under:\n  {run_dir}")


Run ID: 20260922-221848_roberta-base
Outputs will be written under: C:\Users\X\Desktop\X\ALTA\runs\20260922-221848_roberta-base
Using device: cuda

Fine-tuning for task: sentiment


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

{'loss': '0.6925', 'grad_norm': '1.443', 'learning_rate': '4.712e-06', 'epoch': '0.1445'}
{'loss': '0.575', 'grad_norm': '15.56', 'learning_rate': '9.519e-06', 'epoch': '0.289'}
{'loss': '0.2565', 'grad_norm': '23.08', 'learning_rate': '1.433e-05', 'epoch': '0.4335'}
{'loss': '0.2734', 'grad_norm': '12.15', 'learning_rate': '1.913e-05', 'epoch': '0.578'}
{'loss': '0.2572', 'grad_norm': '28', 'learning_rate': '1.956e-05', 'epoch': '0.7225'}
{'loss': '0.2389', 'grad_norm': '23.96', 'learning_rate': '1.903e-05', 'epoch': '0.8671'}
{'eval_loss': '0.2131', 'eval_macro_f1': '0.9287', 'eval_macro_f1_en-AU': '0.9161', 'eval_macro_f1_en-UK': '0.9401', 'eval_weak_macro_f1': '0.9161', 'eval_runtime': '3.637', 'eval_samples_per_second': '208.2', 'eval_steps_per_second': '6.6', 'epoch': '1'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'loss': '0.2803', 'grad_norm': '17.06', 'learning_rate': '1.849e-05', 'epoch': '1.012'}
{'loss': '0.216', 'grad_norm': '32.84', 'learning_rate': '1.796e-05', 'epoch': '1.156'}
{'loss': '0.1497', 'grad_norm': '1.493', 'learning_rate': '1.742e-05', 'epoch': '1.301'}
{'loss': '0.1764', 'grad_norm': '16.82', 'learning_rate': '1.688e-05', 'epoch': '1.445'}
{'loss': '0.2076', 'grad_norm': '3.634', 'learning_rate': '1.635e-05', 'epoch': '1.59'}
{'loss': '0.2065', 'grad_norm': '14.56', 'learning_rate': '1.581e-05', 'epoch': '1.734'}
{'loss': '0.1328', 'grad_norm': '54.11', 'learning_rate': '1.528e-05', 'epoch': '1.879'}
{'eval_loss': '0.2583', 'eval_macro_f1': '0.9392', 'eval_macro_f1_en-AU': '0.9136', 'eval_macro_f1_en-UK': '0.9636', 'eval_weak_macro_f1': '0.9136', 'eval_runtime': '3.59', 'eval_samples_per_second': '210.9', 'eval_steps_per_second': '6.686', 'epoch': '2'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (15 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '422', 'train_samples_per_second': '78.64', 'train_steps_per_second': '4.919', 'train_loss': '0.2475', 'epoch': '5'}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Wrote predictions to C:\Users\X\Desktop\X\ALTA\runs\20260922-221848_roberta-base\answer.csv

Official style scores (weak macro F1) on the public valid set:
  sentiment-en-AU: 0.9161
  sentiment-en-UK: 0.9401
  sarcasm-en-AU: 0.7558
  sarcasm-en-UK: 0.6761
  final score: 0.7961

All metrics, confusion matrices, and models saved under:
  C:\Users\X\Desktop\X\ALTA\runs\20260922-221848_roberta-base
